In [ ]:
"""
آزمون فرض امتیازی اول 
Chi-Square
آسانسور و بازسازی

"""

import pandas as pd
from scipy.stats import chi2_contingency


file_path = "final_data.csv"
df = pd.read_csv(
    file_path,
    usecols=["has_elevator", "is_rebuilt"])
print("Initial rows:", len(df))


df = df.dropna(
    subset=["has_elevator", "is_rebuilt"])

print("Rows after removing missing values:", len(df))


table = pd.crosstab(
    df["has_elevator"],
    df["is_rebuilt"])

print("\nContingency table:")
print(table)


chi2, p_value, dof, expected = chi2_contingency(table)


print("\nChi-Square test")
print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)


alpha = 0.05

if p_value < alpha:

    print("\nReject H0")
    print("There is a significant relationship between the variables.")

else:

    print("\nFail to reject H0")
    print("There is no significant relationship between the variables.")

Initial rows: 1000000
Rows after removing missing values: 381932

Contingency table:
is_rebuilt     False  True 
has_elevator               
False          77996  59420
True          163144  81372

Chi-Square test
Chi-square statistic: 3750.9449096114013
p-value: 0.0
Degrees of freedom: 1

Reject H0
There is a significant relationship between the variables.


In [ ]:
"""
آزمون فرض امتیازی دوم 
Kruskal-Wallis
جهت ساختمان و قیمت
"""
import pandas as pd
from scipy.stats import kruskal


file_path = "final_data.csv"

df = pd.read_csv(
    file_path,
    usecols=["price_value", "building_direction"])

print("Initial rows:", len(df))


df = df.dropna(
    subset=["price_value", "building_direction"])

df["price_value"] = pd.to_numeric(
    df["price_value"],
    errors="coerce")

df = df.dropna(subset=["price_value"])

print("Rows after removing missing values:", len(df))


# محدود کردن داده های پرت با IQR

Q1 = df["price_value"].quantile(0.25)
Q3 = df["price_value"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# جایگزین کردن مقادیر پرت با حد مجاز IQR
df["price_value"] = df["price_value"].clip(
    lower=lower,
    upper=upper)

print("Rows after handling outliers:", len(df))

print("\nBuilding direction:")
print(df["building_direction"].value_counts())


groups = []

for direction in df["building_direction"].unique():

    group = df.loc[
        df["building_direction"] == direction,
        "price_value"]

    groups.append(group)


statistic, p_value = kruskal(*groups)


print("\nKruskal-Wallis test")
print("H statistic:", statistic)
print("p-value:", p_value)


alpha = 0.05

if p_value < alpha:

    print("\nReject H0")
    print("There is a significant difference between the groups.")

else:

    print("\nFail to reject H0")
    print("There is no significant difference between the groups.")


# مقایسه قیمت ملک بر اساس جهت ساختمان

summary = df.groupby("building_direction")["price_value"].agg(
    count="count",
    mean="mean",
    median="median",
    min="min",
    max="max"
).sort_values("median", ascending=False)

print("\nPrice comparison by building direction:")
print(summary)

Initial rows: 1000000
Rows after removing missing values: 202871
Rows after handling outliers: 202871

Building direction:
building_direction
south    100670
north     90073
east       7674
west       4454
Name: count, dtype: int64

Kruskal-Wallis test
H statistic: 2829.6358079302295
p-value: 0.0

Reject H0
There is a significant difference between the groups.

Price comparison by building direction:
                     count          mean        median  min           max
building_direction                                                       
south               100670  5.199437e+09  3.800000e+09  0.0  1.328750e+10
north                90073  4.480585e+09  3.200000e+09  0.0  1.328750e+10
east                  7674  3.874512e+09  2.800000e+09  0.0  1.328750e+10
west                  4454  3.868911e+09  2.780000e+09  0.0  1.328750e+10
